# YOLO26n + ASFF — Clean Full-Training Research Pipeline

Notebook ini adalah pipeline **end-to-end full training** untuk eksperimen **YOLO26n-ASFF** pada dataset IndoTraffic audited.

Prinsip eksperimen:

- baseline kontrol tetap YOLO26n dengan FPN-PAN bawaan;
- konfigurasi ini mengganti mekanisme multi-scale feature fusion pada neck dengan **Adaptive Spatial Feature Fusion (ASFF)**;
- backbone dan detection head YOLO26n dipertahankan;
- input ASFF adalah fitur backbone P3/P4/P5 pada stride 8/16/32;
- setiap target level menerima ketiga feature map setelah spatial/channel alignment;
- ASFF mempelajari tiga bobot **spasial** per lokasi melalui softmax;
- output ASFF diselaraskan menjadi P3=64, P4=128, P5=256 channel agar Detect menerima interface yang sama dengan baseline;
- dataset dikunci pada Roboflow `poltrada-original-raw-2`, Version `2`;
- seed penelitian: `42`, `123`, `2026`;
- default notebook menjalankan **satu full-training seed aktif**, yaitu seed `42`;
- untuk seed berikutnya cukup ganti `active_seed` menjadi `123` atau `2026` tanpa mengubah arsitektur/hyperparameter lain;
- `best.pt` dari masing-masing seed digunakan untuk evaluasi;
- konfigurasi training mengikuti baseline/BiFPN agar comparison tetap terkontrol;
- evaluator memakai pemrosesan image path secara terkontrol agar `eval_batch=1` benar-benar dihormati dan tidak mengulang OOM evaluator lama.

Notebook ini **tidak menjalankan smoke test 1-epoch**. Sebagai gantinya, sebelum full training terdapat hard gate untuk memverifikasi identitas backbone/head, bentuk tensor P3/P4/P5, softmax ASFF, gradient ke spatial-weight logits, dan custom trainer graph.

Evaluasi mengikuti proposal:

1. kinerja keseluruhan: mAP50-95, mAP50, Precision, Recall, F1;
2. per-class AP dan confusion matrix;
3. AP-S/AP-M/AP-L pada kanvas evaluasi 640×640;
4. Parameters, GFLOPs, training time, latency, FPS;
5. diagnostik tambahan: ringkasan bobot spasial ASFF.

Friedman Test tidak dilakukan di notebook satu arsitektur ini. Uji statistik dilakukan setelah FPN-PAN, BiFPN, ASFF, dan AFPN masing-masing memiliki tiga seed lengkap.


## A. Persiapan platform

Aktifkan **GPU** dan **Internet** pada Kaggle. Tambahkan `ROBOFLOW_API_KEY` dan `WANDB_API_KEY` pada Kaggle Secrets.

Versi utama dipin agar lingkungan eksperimen konsisten dengan baseline. PyTorch bawaan Kaggle tidak diganti.


In [ ]:
%pip install -q "ultralytics==8.4.155" "wandb==0.28.1" "roboflow==1.4.0" pycocotools pyyaml pandas matplotlib pillow scipy thop

from importlib.metadata import version

REQUIRED_VERSIONS = {
    "ultralytics": "8.4.155",
    "wandb": "0.28.1",
    "roboflow": "1.4.0",
}

for package, expected in REQUIRED_VERSIONS.items():
    installed = version(package)
    print(f"{package:12s}: {installed} (expected {expected})")
    if installed != expected:
        raise RuntimeError(
            f"Versi {package} tidak sesuai. Restart kernel lalu Run All kembali."
        )


## B. Environment dan direktori kerja


In [ ]:
import gc
import hashlib
import importlib
import json
import os
import platform
import random
import shutil
import sys
import time
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import ultralytics
import wandb
import yaml
from IPython.display import Image as IPyImage, Markdown, display
from PIL import Image

from ultralytics import YOLO, settings
from ultralytics.nn.tasks import DetectionModel

IN_KAGGLE = Path("/kaggle/working").exists()
IN_COLAB = "google.colab" in sys.modules
PLATFORM = "kaggle" if IN_KAGGLE else "colab" if IN_COLAB else "local"

if IN_KAGGLE:
    WORK_ROOT = Path("/kaggle/working/yolo26n_asff_indotraffic")
elif IN_COLAB:
    WORK_ROOT = Path("/content/yolo26n_asff_indotraffic")
else:
    WORK_ROOT = Path("./yolo26n_asff_indotraffic")

WORK_ROOT = WORK_ROOT.resolve()
DATA_ROOT = WORK_ROOT / "dataset"
RUNS_ROOT = WORK_ROOT / "runs"
REPORTS_ROOT = WORK_ROOT / "reports"

for directory in (WORK_ROOT, RUNS_ROOT, REPORTS_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

GPU_AVAILABLE = torch.cuda.is_available()
DEVICE = 0 if GPU_AVAILABLE else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else "Tidak tersedia"
GPU_VRAM_GB = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3
    if GPU_AVAILABLE
    else 0.0
)

ENV_INFO = {
    "platform": PLATFORM,
    "python": sys.version.split()[0],
    "pytorch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "wandb": wandb.__version__,
    "cuda_available": GPU_AVAILABLE,
    "cuda": torch.version.cuda,
    "gpu": GPU_NAME,
    "gpu_vram_gb": round(GPU_VRAM_GB, 2),
}

print(json.dumps(ENV_INFO, indent=2))
print("Work root:", WORK_ROOT)

if not GPU_AVAILABLE:
    raise RuntimeError("CUDA GPU tidak tersedia. Aktifkan GPU sebelum melanjutkan.")

# ============================================================
# FAIRNESS GATE — environment used by executed baseline
# ============================================================
BASELINE_ENV_EXPECTED = {
    "python": "3.12.13",
    "pytorch": "2.10.0+cu128",
    "ultralytics": "8.4.155",
    "wandb": "0.28.1",
    "cuda": "12.8",
    "gpu": "Tesla T4",
}

environment_mismatch = {
    key: {
        "expected": expected,
        "actual": ENV_INFO.get(key),
    }
    for key, expected in BASELINE_ENV_EXPECTED.items()
    if ENV_INFO.get(key) != expected
}

if environment_mismatch:
    raise RuntimeError(
        "Environment berbeda dari baseline yang telah dieksekusi:\n"
        + json.dumps(environment_mismatch, indent=2)
    )

print("BASELINE ENVIRONMENT FAIRNESS GATE: PASSED")

# Matikan integrasi W&B bawaan Ultralytics; notebook melakukan logging eksplisit.
settings.update({"wandb": False})


## C. Konfigurasi eksperimen

Default menjalankan **satu seed per Kaggle run** agar recovery sederhana. Untuk seed berikutnya, ubah hanya `active_seed` menjadi `123` lalu `2026`.

Jangan mengubah hyperparameter di antara seed atau antararsitektur.


In [ ]:
CFG = {
    # Identity
    "architecture": "ASFF",
    "model_name": "YOLO26n-ASFF",
    "model": "yolo26n.pt",

    # Dataset
    "workspace": "ilham-rafiqin",
    "project": "poltrada-original-raw-2",
    "dataset_version": 2,
    "class_names": [
        "Bus",
        "Mobil Penumpang",
        "Pejalan Kaki",
        "Sepeda Motor",
        "Truck",
        "Unmotorized",
    ],

    # Reproducibility
    "seeds": [42, 123, 2026],
    "active_seed": 42,
    "run_all_seeds": False,

    # ASFF design fixed for this research experiment
    "asff_weight_channels": 16,
    "backbone_feature_layers": [4, 6, 10],
    "backbone_feature_channels": [128, 128, 256],
    "detect_feature_channels": [64, 128, 256],

    # Training — same protocol as baseline
    "imgsz": 640,
    "train_batch": 16,
    "epochs": 100,
    "patience": 25,
    "optimizer": "AdamW",
    "lr0": 0.001,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "amp": True,
    "workers": 4,
    "save_period": 1,
    "artifact_period": 5,
    "close_mosaic": 10,
    "flipud": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,

    # Evaluation
    "eval_batch": 1,
    "prediction_conf": 0.001,
    "prediction_iou": 0.7,

    # Controlled efficiency benchmark
    "benchmark_warmup": 50,
    "benchmark_runs": 200,

    # W&B
    "wandb_project": "skripsi-yolo26n-indotraffic",
    "wandb_entity": None,
}

if CFG["active_seed"] not in CFG["seeds"]:
    raise ValueError("active_seed harus merupakan salah satu CFG['seeds'].")

SEEDS_TO_RUN = (
    CFG["seeds"]
    if CFG["run_all_seeds"]
    else [CFG["active_seed"]]
)

print("Architecture :", CFG["model_name"])
print("Dataset      :", f"{CFG['project']} v{CFG['dataset_version']}")
print("Seeds config :", CFG["seeds"])
print("Seeds to run :", SEEDS_TO_RUN)


## D. Secrets dan W&B


In [ ]:
def get_secret(name):
    value = os.getenv(name)
    if value:
        return value

    if IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        except Exception:
            pass

    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            pass

    return None


ROBOFLOW_API_KEY = get_secret("ROBOFLOW_API_KEY")
WANDB_API_KEY = get_secret("WANDB_API_KEY")

if not ROBOFLOW_API_KEY:
    raise RuntimeError("ROBOFLOW_API_KEY tidak ditemukan.")

WANDB_ENABLED = bool(WANDB_API_KEY)

if WANDB_ENABLED:
    wandb.login(key=WANDB_API_KEY, relogin=True)
    print("W&B: aktif")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("W&B: nonaktif; hasil tetap disimpan lokal.")


## E. Download dataset audited Version 2

Notebook menggunakan snapshot final yang sudah diaudit. Folder lokal hanya digunakan sebagai working copy; sumber di Roboflow tidak dimodifikasi.

`data.yaml` tidak diasumsikan berada pada path tertentu. Notebook mencari dan membuat runtime YAML dengan path absolut.


In [ ]:
from roboflow import Roboflow

EXPECTED_SOURCE = {
    "workspace": CFG["workspace"],
    "project": CFG["project"],
    "version": CFG["dataset_version"],
}
SOURCE_MARKER = DATA_ROOT / ".dataset_source.json"

reuse_dataset = False
if DATA_ROOT.exists() and SOURCE_MARKER.exists():
    try:
        previous_source = json.loads(SOURCE_MARKER.read_text(encoding="utf-8"))
        yaml_candidates = list(DATA_ROOT.rglob("data.yaml"))
        reuse_dataset = previous_source == EXPECTED_SOURCE and bool(yaml_candidates)
    except Exception:
        reuse_dataset = False

if reuse_dataset:
    SOURCE_YAML = list(DATA_ROOT.rglob("data.yaml"))[0]
    print("[INFO] Menggunakan exact dataset snapshot yang sudah tersedia:", SOURCE_YAML.parent)
else:
    if DATA_ROOT.exists():
        shutil.rmtree(DATA_ROOT)

    print("[INFO] Download exact Roboflow dataset...")
    print(EXPECTED_SOURCE)

    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    rf_project = (
        rf.workspace(CFG["workspace"])
        .project(CFG["project"])
    )
    downloaded = (
        rf_project
        .version(CFG["dataset_version"])
        .download(
            "yolov8",
            location=str(DATA_ROOT),
            overwrite=True,
        )
    )

    candidates = []
    for root in [DATA_ROOT, Path(downloaded.location)]:
        if root.exists():
            candidates.extend(root.rglob("data.yaml"))
    candidates = list(dict.fromkeys(candidates))

    if not candidates:
        raise FileNotFoundError(
            "Dataset selesai diunduh tetapi data.yaml tidak ditemukan."
        )

    SOURCE_YAML = candidates[0]
    SOURCE_MARKER.parent.mkdir(parents=True, exist_ok=True)
    SOURCE_MARKER.write_text(
        json.dumps(EXPECTED_SOURCE, indent=2),
        encoding="utf-8",
    )

raw_cfg = yaml.safe_load(SOURCE_YAML.read_text(encoding="utf-8"))
names = raw_cfg.get("names", [])

if isinstance(names, dict):
    names = [names[k] for k in sorted(names, key=lambda x: int(x))]
else:
    names = list(names)

if names != CFG["class_names"]:
    raise ValueError(
        "Urutan kelas dataset berbeda.\n"
        f"Expected: {CFG['class_names']}\n"
        f"Found   : {names}"
    )

DATASET_NC = len(names)


def resolve_split(key, aliases=()):
    value = raw_cfg.get(key)
    candidates = []

    if value:
        p = Path(str(value))
        candidates.append(
            p if p.is_absolute()
            else (SOURCE_YAML.parent / p).resolve()
        )

    for alias in aliases:
        candidates.append(
            (SOURCE_YAML.parent / alias / "images").resolve()
        )

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        f"Split '{key}' tidak ditemukan. Kandidat:\n"
        + "\n".join(str(path) for path in candidates)
    )


SPLIT_IMAGES = {
    "train": resolve_split("train", ("train",)),
    "val": resolve_split("val", ("valid", "val")),
    "test": resolve_split("test", ("test",)),
}

runtime_cfg = {
    "path": str(SOURCE_YAML.parent.resolve()),
    "train": str(SPLIT_IMAGES["train"]),
    "val": str(SPLIT_IMAGES["val"]),
    "test": str(SPLIT_IMAGES["test"]),
    "nc": DATASET_NC,
    "names": names,
}

RUNTIME_YAML = WORK_ROOT / "data_runtime.yaml"
RUNTIME_YAML.write_text(
    yaml.safe_dump(runtime_cfg, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

print("Source YAML :", SOURCE_YAML)
print("Classes     :", names)
print("Train       :", SPLIT_IMAGES["train"])
print("Validation  :", SPLIT_IMAGES["val"])
print("Test        :", SPLIT_IMAGES["test"])
print("Runtime YAML:", RUNTIME_YAML)


## F. Normalisasi format label untuk object detection

Dataset penelitian adalah object detection. Baris YOLO bbox lima kolom dipertahankan. Bila export memuat polygon YOLO (`class x1 y1 ...`), polygon dikonversi secara deterministik menjadi **enclosing bounding box**.

File label yang berubah dibackup sebelum ditulis ulang. Jumlah objek, split, class ID, dan image tidak diubah.


In [ ]:
# ============================================================
# G. NORMALIZE LABEL FORMAT FOR OBJECT DETECTION
# ============================================================
#
# Tujuan:
# - YOLO bbox (5 kolom) -> dibiarkan
# - YOLO polygon        -> dikonversi menjadi bounding box
# - format lain         -> dihentikan
#
# Tidak mengubah split, class ID, atau image.
# ============================================================

from pathlib import Path
import shutil
import pandas as pd


BACKUP_ROOT = (
    REPORTS_ROOT
    / "label_backup_before_polygon_conversion"
)

BACKUP_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


conversion_rows = []


def polygon_to_bbox(parts):
    """
    Convert:
        class x1 y1 x2 y2 ... xn yn

    menjadi:
        class xc yc width height
    """

    class_id = parts[0]

    coords = list(
        map(
            float,
            parts[1:]
        )
    )

    # Polygon minimal 3 titik:
    # class + 6 coordinate = 7 columns
    if len(coords) < 6:
        raise ValueError(
            "Polygon memiliki kurang dari 3 titik."
        )

    if len(coords) % 2 != 0:
        raise ValueError(
            "Jumlah koordinat polygon bukan pasangan x,y."
        )


    xs = coords[0::2]
    ys = coords[1::2]


    # Pastikan normalized coordinates masuk akal
    if not all(
        0 <= value <= 1
        for value in xs + ys
    ):
        raise ValueError(
            "Koordinat polygon berada di luar rentang [0,1]."
        )


    x_min = min(xs)
    x_max = max(xs)

    y_min = min(ys)
    y_max = max(ys)


    width = x_max - x_min
    height = y_max - y_min


    if width <= 0 or height <= 0:
        raise ValueError(
            "Polygon menghasilkan bbox dengan ukuran <= 0."
        )


    x_center = (
        x_min + x_max
    ) / 2

    y_center = (
        y_min + y_max
    ) / 2


    return (
        f"{class_id} "
        f"{x_center:.10f} "
        f"{y_center:.10f} "
        f"{width:.10f} "
        f"{height:.10f}"
    )


# ============================================================
# Find labels that need normalization
# ============================================================

converted_files = 0
converted_objects = 0


for split, image_dir in SPLIT_IMAGES.items():

    image_dir = Path(
        image_dir
    )

    label_dir = (
        image_dir.parent
        / "labels"
    )


    if not label_dir.exists():
        continue


    for label_path in sorted(
        label_dir.rglob("*.txt")
    ):

        original_text = (
            label_path.read_text(
                encoding="utf-8"
            )
        )

        lines = (
            original_text
            .splitlines()
        )


        new_lines = []

        file_changed = False


        for line_number, raw_line in enumerate(
            lines,
            start=1,
        ):

            line = raw_line.strip()


            if not line:
                continue


            parts = line.split()


            # ------------------------------------------------
            # Standard YOLO detection
            # ------------------------------------------------

            if len(parts) == 5:

                new_lines.append(
                    line
                )

                continue


            # ------------------------------------------------
            # YOLO polygon:
            # class + >= 3 coordinate pairs
            # ------------------------------------------------

            if (
                len(parts) >= 7
                and len(parts) % 2 == 1
            ):

                bbox_line = polygon_to_bbox(
                    parts
                )

                new_lines.append(
                    bbox_line
                )

                file_changed = True
                converted_objects += 1


                conversion_rows.append({
                    "split": split,
                    "label": str(label_path),
                    "line": line_number,
                    "original_columns": len(parts),
                    "action": "polygon_to_bbox",
                    "original": line,
                    "converted": bbox_line,
                })

                continue


            # ------------------------------------------------
            # Unknown format
            # ------------------------------------------------

            raise RuntimeError(
                "Format label tidak dikenali:\n"
                f"File : {label_path}\n"
                f"Line : {line_number}\n"
                f"Cols : {len(parts)}\n"
                f"Raw  : {line}"
            )


        # ====================================================
        # Write only affected files
        # ====================================================

        if file_changed:

            relative_path = (
                label_path
                .relative_to(
                    label_dir
                )
            )


            backup_path = (
                BACKUP_ROOT
                / split
                / relative_path
            )


            backup_path.parent.mkdir(
                parents=True,
                exist_ok=True,
            )


            shutil.copy2(
                label_path,
                backup_path,
            )


            label_path.write_text(
                "\n".join(
                    new_lines
                ) + "\n",
                encoding="utf-8",
            )


            converted_files += 1


# ============================================================
# Conversion report
# ============================================================

CONVERSION_DF = pd.DataFrame(
    conversion_rows
)


CONVERSION_REPORT = (
    REPORTS_ROOT
    / "polygon_to_bbox_conversion.csv"
)


if not CONVERSION_DF.empty:

    CONVERSION_DF.to_csv(
        CONVERSION_REPORT,
        index=False,
    )


print("=" * 70)
print("LABEL NORMALIZATION COMPLETE")
print("=" * 70)

print(
    "Files converted  :",
    converted_files,
)

print(
    "Objects converted:",
    converted_objects,
)

print(
    "Backup directory :",
    BACKUP_ROOT,
)

print(
    "Conversion report:",
    CONVERSION_REPORT,
)


if not CONVERSION_DF.empty:

    display(
        CONVERSION_DF[
            [
                "split",
                "label",
                "line",
                "original_columns",
                "action",
            ]
        ]
    )


## G. Audit dataset dan provenance gate

Audit menghentikan training jika ada image corrupt, label hilang, class ID invalid, koordinat invalid, bounding box keluar citra di luar toleransi floating point, atau duplicate image lintas split.

Selain itu, snapshot V2 diverifikasi terhadap jumlah citra, anotasi, dan empty label yang sudah dikonfirmasi pada audit RAW-vs-AUDITED.


In [ ]:
# ============================================================
# G. DATASET AUDIT
# ============================================================

from pathlib import Path
import hashlib

import pandas as pd
from PIL import Image


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

IMAGE_EXTS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

# Toleransi floating-point untuk bbox yang menyentuh tepi citra.
# Dataset kita memiliki overflow maksimum sekitar 5e-6.
BOUNDARY_EPS = 1e-5

NON_BLOCKING_STATUS = {
    "ok",
    "empty_label",
}


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def label_dir_for(image_dir: Path) -> Path:
    """Dapatkan direktori label pasangan dari direktori images."""

    if image_dir.name == "images":
        return image_dir.parent / "labels"

    return Path(
        str(image_dir).replace(
            "images",
            "labels",
        )
    )


def sha256_file(path: Path) -> str:
    """Hitung SHA256 tanpa membaca seluruh file ke RAM sekaligus."""

    hasher = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            hasher.update(chunk)

    return hasher.hexdigest()


def validate_label_file(
    label_path: Path,
    num_classes: int,
    boundary_eps: float = BOUNDARY_EPS,
):
    """
    Validasi satu file label YOLO detection.

    Returns
    -------
    status : str
    object_count : int
    issue : dict | None
    """

    # --------------------------------------------------------
    # Label tidak tersedia
    # --------------------------------------------------------

    if not label_path.exists():
        return (
            "missing_label",
            0,
            {
                "issue": "missing_label",
                "line": None,
                "raw": None,
            },
        )


    # --------------------------------------------------------
    # Read label
    # --------------------------------------------------------

    lines = [
        line.strip()
        for line in label_path.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]


    if not lines:
        return (
            "empty_label",
            0,
            None,
        )


    object_count = 0


    # --------------------------------------------------------
    # Validate each YOLO annotation
    # --------------------------------------------------------

    for line_no, line in enumerate(
        lines,
        start=1,
    ):

        parts = line.split()


        # YOLO detection harus:
        # class xc yc width height
        if len(parts) != 5:

            return (
                f"invalid_label_line_{line_no}:"
                "jumlah kolom bukan 5",

                object_count,

                {
                    "issue": "column_count",
                    "line": line_no,
                    "n_columns": len(parts),
                    "raw": line,
                },
            )


        try:

            cls, xc, yc, bw, bh = map(
                float,
                parts,
            )

        except ValueError:

            return (
                f"invalid_label_line_{line_no}:"
                "nilai bukan numerik",

                object_count,

                {
                    "issue": "non_numeric",
                    "line": line_no,
                    "raw": line,
                },
            )


        # ----------------------------------------------------
        # Class ID
        # ----------------------------------------------------

        if (
            not cls.is_integer()
            or not 0 <= int(cls) < num_classes
        ):

            return (
                f"invalid_label_line_{line_no}:"
                "class id invalid",

                object_count,

                {
                    "issue": "invalid_class",
                    "line": line_no,
                    "raw": line,
                },
            )


        # ----------------------------------------------------
        # Basic normalized-coordinate validation
        # ----------------------------------------------------

        if not (
            0 <= xc <= 1
            and 0 <= yc <= 1
            and 0 < bw <= 1
            and 0 < bh <= 1
        ):

            return (
                f"invalid_label_line_{line_no}:"
                "koordinat invalid",

                object_count,

                {
                    "issue": "invalid_coordinate",
                    "line": line_no,
                    "raw": line,
                },
            )


        # ----------------------------------------------------
        # Bounding-box boundaries
        # ----------------------------------------------------

        x1 = xc - (bw / 2)
        y1 = yc - (bh / 2)

        x2 = xc + (bw / 2)
        y2 = yc + (bh / 2)


        overflow = max(
            0.0,
            -x1,
            -y1,
            x2 - 1.0,
            y2 - 1.0,
        )


        # Hanya dianggap invalid jika melebihi
        # toleransi floating-point.
        if overflow > boundary_eps:

            return (
                f"invalid_label_line_{line_no}:"
                "box keluar citra",

                object_count,

                {
                    "issue": "out_of_bounds",
                    "line": line_no,
                    "overflow": overflow,
                    "raw": line,
                },
            )


        object_count += 1


    return (
        "ok",
        object_count,
        None,
    )


# ============================================================
# Run audit
# ============================================================

audit_rows = []
issue_rows = []

seen_hashes = {}


for split, image_dir in SPLIT_IMAGES.items():

    image_dir = Path(image_dir)

    label_dir = label_dir_for(
        image_dir
    )


    image_paths = sorted(
        path
        for path in image_dir.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in IMAGE_EXTS
        )
    )


    print(
        f"[AUDIT] {split}: "
        f"{len(image_paths)} images"
    )


    for image_path in image_paths:

        relative_label = (
            image_path
            .relative_to(image_dir)
            .with_suffix(".txt")
        )

        label_path = (
            label_dir
            / relative_label
        )


        row = {
            "split": split,
            "image": str(image_path),
            "label": str(label_path),
            "objects": 0,
            "status": "ok",
            "width": None,
            "height": None,
        }


        # ----------------------------------------------------
        # Validate image
        # ----------------------------------------------------

        try:

            with Image.open(
                image_path
            ) as image:

                image.verify()


            with Image.open(
                image_path
            ) as image:

                row["width"], row["height"] = (
                    image.size
                )


        except Exception as exc:

            row["status"] = (
                f"corrupt_image:"
                f"{type(exc).__name__}"
            )


        # ----------------------------------------------------
        # Validate annotation
        # ----------------------------------------------------

        if row["status"] == "ok":

            (
                status,
                object_count,
                issue,
            ) = validate_label_file(
                label_path=label_path,
                num_classes=len(names),
            )


            row["status"] = status
            row["objects"] = object_count


            if issue is not None:

                issue_rows.append({
                    "split": split,
                    "image": str(image_path),
                    "label": str(label_path),
                    **issue,
                })


        # ----------------------------------------------------
        # Hashes
        # ----------------------------------------------------

        image_hash = sha256_file(
            image_path
        )


        label_hash = (
            sha256_file(label_path)
            if label_path.exists()
            else None
        )


        row["sha256"] = image_hash
        row["label_sha256"] = label_hash


        seen_hashes.setdefault(
            image_hash,
            [],
        ).append(
            (
                split,
                str(image_path),
            )
        )


        audit_rows.append(
            row
        )


# ============================================================
# Build audit tables
# ============================================================

AUDIT_DF = pd.DataFrame(
    audit_rows
)

ISSUES_DF = pd.DataFrame(
    issue_rows
)


# ------------------------------------------------------------
# Cross-split duplicate detection
# ------------------------------------------------------------

duplicate_leaks = {
    digest: occurrences

    for digest, occurrences
    in seen_hashes.items()

    if len({
        split
        for split, _
        in occurrences
    }) > 1
}


# ------------------------------------------------------------
# Dataset fingerprint
# ------------------------------------------------------------

fingerprint_payload = "\n".join(
    sorted(
        (
            f"{row['split']}|"
            f"{Path(row['image']).name}|"
            f"{row['sha256']}|"
            f"{row['label_sha256']}"
        )

        for row in audit_rows
    )
)


DATASET_FINGERPRINT = (
    hashlib
    .sha256(
        fingerprint_payload.encode(
            "utf-8"
        )
    )
    .hexdigest()
)


# ============================================================
# Save audit report
# ============================================================

AUDIT_DF.to_csv(
    REPORTS_ROOT
    / "dataset_audit.csv",
    index=False,
)


if not ISSUES_DF.empty:

    ISSUES_DF.to_csv(
        REPORTS_ROOT
        / "dataset_audit_issues.csv",
        index=False,
    )


# ============================================================
# Summary
# ============================================================

summary = (
    AUDIT_DF
    .groupby(
        [
            "split",
            "status",
        ],
        as_index=False,
    )
    .agg(
        images=("image", "count"),
        objects=("objects", "sum"),
    )
)


display(
    summary
)


print()
print(
    "Dataset fingerprint:",
    DATASET_FINGERPRINT,
)

print(
    "Duplicate cross-split:",
    len(duplicate_leaks),
)

print(
    "Empty labels:",
    int(
        (
            AUDIT_DF["status"]
            == "empty_label"
        ).sum()
    ),
)


# ============================================================
# Blocking issues
# ============================================================

blocking = AUDIT_DF[
    ~AUDIT_DF["status"].isin(
        NON_BLOCKING_STATUS
    )
].copy()


print()
print(
    "Blocking rows:",
    len(blocking),
)


if len(blocking):

    display(
        blocking[
            [
                "split",
                "image",
                "label",
                "status",
            ]
        ]
    )


    if not ISSUES_DF.empty:

        print()
        print(
            "Detail masalah anotasi:"
        )

        display(
            ISSUES_DF
        )


if duplicate_leaks:

    print()
    print(
        "Duplicate images across splits:"
    )

    for digest, occurrences in (
        duplicate_leaks.items()
    ):

        print(
            digest,
            occurrences,
        )


# ============================================================
# Final gate
# ============================================================

if (
    len(blocking) > 0
    or len(duplicate_leaks) > 0
):

    raise RuntimeError(
        "Audit belum lolos. "
        "Lihat blocking dan ISSUES_DF "
        "di atas sebelum training."
    )


print()
print("=" * 70)
print("DATASET AUDIT PASSED")
print("=" * 70)



# ============================================================
# Exact audited-V2 provenance gate
# ============================================================

EXPECTED_AUDITED_V2 = {
    "train": {"images": 6775, "annotations": 125108, "empty_labels": 16},
    "val":   {"images": 1790, "annotations": 32076,  "empty_labels": 2},
    "test":  {"images": 913,  "annotations": 16158,  "empty_labels": 9},
}

actual_rows = []

for split, expected in EXPECTED_AUDITED_V2.items():
    part = AUDIT_DF[AUDIT_DF["split"] == split]

    actual = {
        "split": split,
        "images": int(len(part)),
        "annotations": int(part["objects"].sum()),
        "empty_labels": int((part["status"] == "empty_label").sum()),
    }
    actual_rows.append(actual)

    assert actual["images"] == expected["images"], (
        f"{split}: image count {actual['images']} != {expected['images']}"
    )
    assert actual["annotations"] == expected["annotations"], (
        f"{split}: annotation count {actual['annotations']} != {expected['annotations']}"
    )
    assert actual["empty_labels"] == expected["empty_labels"], (
        f"{split}: empty labels {actual['empty_labels']} != {expected['empty_labels']}"
    )

PROVENANCE_DF = pd.DataFrame(actual_rows)
display(PROVENANCE_DF)

print()
print("DATASET PROVENANCE GATE PASSED")
print("Dataset fingerprint:", DATASET_FINGERPRINT)

# Exact fingerprint observed in the executed FPN-PAN baseline notebook
# after the same deterministic polygon -> bbox normalization.
BASELINE_DATASET_FINGERPRINT = (
    "970d28b5cd507f8dcd054edee478783fa1bf45eb9bac10263896d98f4a96f7cd"
)

assert DATASET_FINGERPRINT == BASELINE_DATASET_FINGERPRINT, (
    "Dataset fingerprint berbeda dari baseline final. "
    f"Expected {BASELINE_DATASET_FINGERPRINT}, got {DATASET_FINGERPRINT}"
)

print("BASELINE DATASET FINGERPRINT FAIRNESS GATE: PASSED")


## H. Implementasi ASFF

Hasil inspeksi YOLO26n yang sudah dikunci pada notebook baseline/BiFPN menunjukkan:

- P3/8 dari layer 4: `128 × 80 × 80`;
- P4/16 dari layer 6: `128 × 40 × 40`;
- P5/32 dari layer 10: `256 × 20 × 20`.

Adaptasi ASFF di notebook ini mempertahankan tiga feature map tersebut sebagai input langsung dari backbone. Untuk setiap target scale (P3, P4, P5), ketiga feature map disejajarkan resolusi dan channel-nya, kemudian ASFF menghasilkan tiga spatial logits. `softmax(dim=1)` memastikan bobot P3/P4/P5 pada setiap posisi berjumlah satu.

Output ASFF tetap `64/128/256` channel agar detection head YOLO26n menerima interface yang sama dengan baseline.

Custom module disimpan sebagai `asff_modules.py` dan ikut diunggah bersama checkpoint agar model dapat direload pada session lain.


In [ ]:
MODULE_PATH = WORK_ROOT / "asff_modules.py"

MODULE_SOURCE = r'''
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBNAct(nn.Module):
    def __init__(self, c1, c2, kernel_size=1, stride=1):
        super().__init__()
        padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv2d(
                c1,
                c2,
                kernel_size,
                stride,
                padding,
                bias=False,
            ),
            nn.BatchNorm2d(c2),
            nn.SiLU(),
        )

    def forward(self, x):
        return self.block(x)


class ASFFScale(nn.Module):
    """
    ASFF fusion untuk satu target scale.

    target_level:
        0 -> P3 / stride 8
        1 -> P4 / stride 16
        2 -> P5 / stride 32

    Bobot ASFF berbentuk [B, 3, H, W], sehingga setiap lokasi
    mempunyai alpha/beta/gamma sendiri untuk sumber P3/P4/P5.
    Bobot dibagi pada seluruh channel di lokasi yang sama.
    """

    def __init__(
        self,
        target_level,
        in_channels=(128, 128, 256),
        out_channels=(64, 128, 256),
        weight_channels=16,
    ):
        super().__init__()

        self.target_level = int(target_level)

        c3, c4, c5 = in_channels
        o3, o4, o5 = out_channels

        if self.target_level == 0:
            c = o3

            self.p3_resize = ConvBNAct(c3, c, 1)
            self.p4_resize = ConvBNAct(c4, c, 1)
            self.p5_resize = ConvBNAct(c5, c, 1)

        elif self.target_level == 1:
            c = o4

            # P3: 80x80 -> 40x40
            self.p3_resize = ConvBNAct(
                c3, c, kernel_size=3, stride=2
            )

            # P4 already 128x40x40
            self.p4_resize = nn.Identity()

            # P5: 256 -> 128, lalu 20x20 -> 40x40
            self.p5_resize = ConvBNAct(c5, c, 1)

        elif self.target_level == 2:
            c = o5

            # P3: 80x80 -> 40x40 -> 20x20
            self.p3_pre = nn.MaxPool2d(
                kernel_size=3,
                stride=2,
                padding=1,
            )
            self.p3_resize = ConvBNAct(
                c3, c, kernel_size=3, stride=2
            )

            # P4: 40x40 -> 20x20
            self.p4_resize = ConvBNAct(
                c4, c, kernel_size=3, stride=2
            )

            # P5 already 256x20x20
            self.p5_resize = nn.Identity()

        else:
            raise ValueError("target_level harus 0, 1, atau 2.")

        self.inter_channels = c

        # Compressed representations used to predict 3 spatial weights.
        self.weight_p3 = ConvBNAct(c, weight_channels, 1)
        self.weight_p4 = ConvBNAct(c, weight_channels, 1)
        self.weight_p5 = ConvBNAct(c, weight_channels, 1)

        self.weight_levels = nn.Conv2d(
            weight_channels * 3,
            3,
            kernel_size=1,
            stride=1,
            padding=0,
        )

        # Post-fusion refinement.
        self.expand = ConvBNAct(
            c, c, kernel_size=3, stride=1
        )

    def _resize_features(self, p3, p4, p5):
        if self.target_level == 0:
            x3 = self.p3_resize(p3)
            target_size = x3.shape[-2:]

            x4 = F.interpolate(
                self.p4_resize(p4),
                size=target_size,
                mode="nearest",
            )
            x5 = F.interpolate(
                self.p5_resize(p5),
                size=target_size,
                mode="nearest",
            )

        elif self.target_level == 1:
            x3 = self.p3_resize(p3)
            x4 = self.p4_resize(p4)
            target_size = x4.shape[-2:]

            x5 = F.interpolate(
                self.p5_resize(p5),
                size=target_size,
                mode="nearest",
            )

            if x3.shape[-2:] != target_size:
                raise RuntimeError(
                    "P3->P4 spatial alignment gagal."
                )

        else:
            x3 = self.p3_resize(self.p3_pre(p3))
            x4 = self.p4_resize(p4)
            x5 = self.p5_resize(p5)
            target_size = x5.shape[-2:]

            if x3.shape[-2:] != target_size:
                raise RuntimeError(
                    "P3->P5 spatial alignment gagal."
                )
            if x4.shape[-2:] != target_size:
                raise RuntimeError(
                    "P4->P5 spatial alignment gagal."
                )

        if not (
            x3.shape == x4.shape == x5.shape
        ):
            raise RuntimeError(
                "ASFF aligned feature shapes tidak identik: "
                f"{x3.shape}, {x4.shape}, {x5.shape}"
            )

        return x3, x4, x5

    def forward(
        self,
        p3,
        p4,
        p5,
        return_weights=False,
    ):
        x3, x4, x5 = self._resize_features(
            p3, p4, p5
        )

        weight_features = torch.cat(
            [
                self.weight_p3(x3),
                self.weight_p4(x4),
                self.weight_p5(x5),
            ],
            dim=1,
        )

        logits = self.weight_levels(weight_features)
        weights = F.softmax(logits, dim=1)

        fused = (
            x3 * weights[:, 0:1]
            + x4 * weights[:, 1:2]
            + x5 * weights[:, 2:3]
        )

        output = self.expand(fused)

        if return_weights:
            return output, weights

        return output


class ASFFNeck(nn.Module):
    """
    Three-level ASFF neck adaptation for YOLO26n.

    Backbone input:
        P3: 128 x 80 x 80
        P4: 128 x 40 x 40
        P5: 256 x 20 x 20

    Detect output:
        P3:  64 x 80 x 80
        P4: 128 x 40 x 40
        P5: 256 x 20 x 20
    """

    def __init__(
        self,
        in_channels=(128, 128, 256),
        out_channels=(64, 128, 256),
        weight_channels=16,
    ):
        super().__init__()

        self.asff_p3 = ASFFScale(
            target_level=0,
            in_channels=in_channels,
            out_channels=out_channels,
            weight_channels=weight_channels,
        )
        self.asff_p4 = ASFFScale(
            target_level=1,
            in_channels=in_channels,
            out_channels=out_channels,
            weight_channels=weight_channels,
        )
        self.asff_p5 = ASFFScale(
            target_level=2,
            in_channels=in_channels,
            out_channels=out_channels,
            weight_channels=weight_channels,
        )

    def forward(self, features):
        if len(features) != 3:
            raise ValueError(
                "ASFFNeck membutuhkan P3, P4, dan P5."
            )

        p3, p4, p5 = features

        return [
            self.asff_p3(p3, p4, p5),
            self.asff_p4(p3, p4, p5),
            self.asff_p5(p3, p4, p5),
        ]

    def forward_with_weights(self, features):
        if len(features) != 3:
            raise ValueError(
                "ASFFNeck membutuhkan P3, P4, dan P5."
            )

        p3, p4, p5 = features

        out_p3, w_p3 = self.asff_p3(
            p3, p4, p5, return_weights=True
        )
        out_p4, w_p4 = self.asff_p4(
            p3, p4, p5, return_weights=True
        )
        out_p5, w_p5 = self.asff_p5(
            p3, p4, p5, return_weights=True
        )

        return (
            [out_p3, out_p4, out_p5],
            {
                "P3": w_p3,
                "P4": w_p4,
                "P5": w_p5,
            },
        )


class FeatureSelect(nn.Module):
    def __init__(self, index):
        super().__init__()
        self.index = index

    def forward(self, features):
        return features[self.index]
'''

MODULE_PATH.write_text(
    MODULE_SOURCE,
    encoding="utf-8",
)

if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

importlib.invalidate_caches()

import asff_modules
asff_modules = importlib.reload(asff_modules)

from asff_modules import (
    ASFFNeck,
    ASFFScale,
    FeatureSelect,
)

ASFF_MODULE_SHA256 = hashlib.sha256(
    MODULE_PATH.read_bytes()
).hexdigest()

print("Custom module:", MODULE_PATH)
print("ASFF module SHA256:", ASFF_MODULE_SHA256)


## I. Build dan verifikasi arsitektur YOLO26n-ASFF

Sebelum training, notebook melakukan hard gate:

- baseline YOLO26n masih mempunyai source Detect `[16, 19, 22]`;
- feature backbone P3/P4/P5 tetap `128/128/256` channel pada stride `8/16/32`;
- pretrained-compatible backbone/head dimuat sama seperti baseline;
- backbone layer `0–10` dipertahankan;
- FPN-PAN diganti oleh `ASFFNeck`;
- detection head enam kelas dipertahankan dan menerima `64×80×80`, `128×40×40`, `256×20×20`;
- setiap ASFF target scale menghasilkan tiga weight map dengan softmax sum = 1;
- gradient harus mencapai convolution pembentuk spatial logits;
- tabel arsitektur disimpan untuk reproducibility.


In [ ]:
def state_dict_identical(module_a, module_b):
    state_a = module_a.state_dict()
    state_b = module_b.state_dict()

    if state_a.keys() != state_b.keys():
        return False

    return all(
        torch.equal(state_a[key].cpu(), state_b[key].cpu())
        for key in state_a
    )


def set_graph_metadata(module, index, source):
    module.i = index
    module.f = source
    module.type = module.__class__.__name__
    module.np = sum(p.numel() for p in module.parameters())
    return module


def capture_layer_shapes(core, layer_indices, imgsz):
    captured = {}
    handles = []

    def make_hook(index):
        def hook(module, inputs, output):
            if torch.is_tensor(output):
                captured[index] = tuple(output.shape)
        return hook

    for index in layer_indices:
        handles.append(
            core.model[index].register_forward_hook(
                make_hook(index)
            )
        )

    device = torch.device("cuda:0")
    core = core.to(device).eval()
    dummy = torch.zeros(1, 3, imgsz, imgsz, device=device)

    with torch.inference_mode():
        core(dummy)

    for handle in handles:
        handle.remove()

    return captured


def build_base_6_core():
    """
    Bangun YOLO26n dengan nc=6 lalu transfer semua pretrained weight
    yang kompatibel. Target class names ditetapkan sebelum load agar
    perilaku remapping class row sama dengan baseline yang sudah dieksekusi.
    """
    pretrained_yolo = YOLO(CFG["model"])
    pretrained_core = pretrained_yolo.model

    base_6_core = DetectionModel(
        cfg=deepcopy(pretrained_core.yaml),
        ch=3,
        nc=DATASET_NC,
        verbose=False,
    )

    base_6_core.names = {
        i: name
        for i, name in enumerate(names)
    }

    pretrained_names = {
        str(name).strip().lower()
        for name in pretrained_core.names.values()
    }
    target_names = {
        str(name).strip().lower()
        for name in base_6_core.names.values()
    }
    overlapping_names = sorted(
        pretrained_names & target_names
    )

    assert overlapping_names == ["bus", "truck"], (
        "Unexpected pretrained/target class overlap: "
        f"{overlapping_names}"
    )

    base_6_core.load(
        pretrained_core,
        verbose=True,
    )

    return base_6_core, pretrained_yolo


def build_final_asff_core(base_core):
    core = deepcopy(base_core)
    old_layers = list(base_core.model)

    backbone = [
        deepcopy(layer)
        for layer in old_layers[:11]
    ]

    asff = set_graph_metadata(
        ASFFNeck(
            in_channels=tuple(
                CFG["backbone_feature_channels"]
            ),
            out_channels=tuple(
                CFG["detect_feature_channels"]
            ),
            weight_channels=CFG[
                "asff_weight_channels"
            ],
        ),
        index=11,
        source=CFG["backbone_feature_layers"],
    )

    select_p3 = set_graph_metadata(
        FeatureSelect(0), index=12, source=11
    )
    select_p4 = set_graph_metadata(
        FeatureSelect(1), index=13, source=11
    )
    select_p5 = set_graph_metadata(
        FeatureSelect(2), index=14, source=11
    )

    detect = deepcopy(old_layers[-1])
    detect.i = 15
    detect.f = [12, 13, 14]
    detect.type = detect.__class__.__name__
    detect.np = sum(
        p.numel()
        for p in detect.parameters()
    )

    core.model = nn.Sequential(
        *(
            backbone
            + [
                asff,
                select_p3,
                select_p4,
                select_p5,
                detect,
            ]
        )
    )

    old_backbone_save = [
        idx
        for idx in getattr(base_core, "save", [])
        if idx <= 10
    ]

    core.save = sorted(
        set(
            old_backbone_save
            + [4, 6, 10, 11, 12, 13, 14]
        )
    )

    core.names = deepcopy(base_core.names)
    core.stride = deepcopy(base_core.stride)

    return core


def build_seeded_asff_template(seed):
    """
    Setiap seed membangun ulang ASFF dan class-specific parts yang
    memang random-initialized, sementara pretrained-compatible
    backbone/head weights selalu berasal dari yolo26n.pt yang sama.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    base_6_core, pretrained_wrapper = build_base_6_core()
    custom_core = build_final_asff_core(base_6_core)

    del pretrained_wrapper
    return custom_core, base_6_core


# ============================================================
# Diagnostic gate against the pinned YOLO26n build
# ============================================================

PRETRAINED_DIAGNOSTIC = YOLO(CFG["model"])
PRETRAINED_DIAGNOSTIC_CORE = PRETRAINED_DIAGNOSTIC.model

assert PRETRAINED_DIAGNOSTIC_CORE.model[-1].f == [16, 19, 22], (
    "Struktur YOLO26n berubah: Detect source tidak lagi [16,19,22]."
)

backbone_shapes = capture_layer_shapes(
    PRETRAINED_DIAGNOSTIC_CORE,
    CFG["backbone_feature_layers"],
    CFG["imgsz"],
)

EXPECTED_BACKBONE_SHAPES = {
    4: (1, 128, 80, 80),
    6: (1, 128, 40, 40),
    10: (1, 256, 20, 20),
}

assert backbone_shapes == EXPECTED_BACKBONE_SHAPES, (
    f"Backbone feature shape berubah: {backbone_shapes}"
)

ASFF_CORE_TEMPLATE, DIAGNOSTIC_BASE_6_CORE = (
    build_seeded_asff_template(CFG["active_seed"])
)
ASFF_CORE_TEMPLATE = ASFF_CORE_TEMPLATE.cpu()

assert ASFF_CORE_TEMPLATE.model[-1].nc == DATASET_NC

# Backbone dan Detect identik dengan 6-class base tepat sebelum neck diganti.
for index in range(11):
    assert state_dict_identical(
        DIAGNOSTIC_BASE_6_CORE.model[index],
        ASFF_CORE_TEMPLATE.model[index],
    ), f"Backbone layer {index} berubah."

assert state_dict_identical(
    DIAGNOSTIC_BASE_6_CORE.model[-1],
    ASFF_CORE_TEMPLATE.model[-1],
), "Detection head berubah saat integrasi ASFF."

assert isinstance(
    ASFF_CORE_TEMPLATE.model[11],
    ASFFNeck,
)


# ============================================================
# Forward gate: Detect interface must remain identical
# ============================================================

test_core = deepcopy(
    ASFF_CORE_TEMPLATE
).to("cuda").eval()

captured_detect_shapes = []


def detect_pre_hook(module, inputs):
    captured_detect_shapes.clear()
    captured_detect_shapes.extend(
        tuple(feature.shape)
        for feature in inputs[0]
    )


handle = test_core.model[-1].register_forward_pre_hook(
    detect_pre_hook
)

with torch.inference_mode():
    test_core(
        torch.zeros(
            1,
            3,
            CFG["imgsz"],
            CFG["imgsz"],
            device="cuda",
        )
    )

handle.remove()

EXPECTED_DETECT_INPUTS = [
    (1, 64, 80, 80),
    (1, 128, 40, 40),
    (1, 256, 20, 20),
]

assert captured_detect_shapes == EXPECTED_DETECT_INPUTS, (
    f"Detect input mismatch: {captured_detect_shapes}"
)


# ============================================================
# ASFF-specific softmax + gradient gates
# ============================================================

neck_test = ASFFNeck(
    in_channels=tuple(
        CFG["backbone_feature_channels"]
    ),
    out_channels=tuple(
        CFG["detect_feature_channels"]
    ),
    weight_channels=CFG[
        "asff_weight_channels"
    ],
).to("cuda")

raw_features = [
    torch.randn(
        2, 128, 80, 80, device="cuda"
    ),
    torch.randn(
        2, 128, 40, 40, device="cuda"
    ),
    torch.randn(
        2, 256, 20, 20, device="cuda"
    ),
]

neck_test.eval()

with torch.no_grad():
    neck_outputs, spatial_weights = (
        neck_test.forward_with_weights(
            raw_features
        )
    )

EXPECTED_NECK_OUTPUTS = [
    (2, 64, 80, 80),
    (2, 128, 40, 40),
    (2, 256, 20, 20),
]

assert [
    tuple(output.shape)
    for output in neck_outputs
] == EXPECTED_NECK_OUTPUTS

for target_level, weight_map in spatial_weights.items():
    assert weight_map.shape[1] == 3

    sum_error = (
        weight_map.sum(dim=1) - 1.0
    ).abs().max().item()

    assert sum_error < 1e-5, (
        target_level,
        sum_error,
    )

    assert torch.isfinite(weight_map).all()
    assert weight_map.min().item() >= 0.0
    assert weight_map.max().item() <= 1.0

    print(
        target_level,
        "weight shape:",
        tuple(weight_map.shape),
        "max softmax sum error:",
        f"{sum_error:.3e}",
        "source mean:",
        weight_map.mean(
            dim=(0, 2, 3)
        ).cpu().numpy().round(4),
    )


neck_test.train()
neck_test.zero_grad(set_to_none=True)

neck_outputs = neck_test(raw_features)

gradient_loss = sum(
    output.square().mean()
    for output in neck_outputs
)

gradient_loss.backward()

weight_logit_modules = {
    "P3 target": neck_test.asff_p3.weight_levels,
    "P4 target": neck_test.asff_p4.weight_levels,
    "P5 target": neck_test.asff_p5.weight_levels,
}

for label, module in weight_logit_modules.items():
    gradient = module.weight.grad

    assert gradient is not None
    assert torch.isfinite(gradient).all()
    assert gradient.abs().sum().item() > 0

    print(
        label,
        "weight-logit grad mean:",
        f"{gradient.abs().mean().item():.6e}",
    )


# ============================================================
# Architecture report
# ============================================================

architecture_rows = []

for index, layer in enumerate(
    ASFF_CORE_TEMPLATE.model
):
    architecture_rows.append(
        {
            "index": index,
            "from": str(
                getattr(layer, "f", None)
            ),
            "module": layer.__class__.__name__,
            "parameters": sum(
                p.numel()
                for p in layer.parameters()
            ),
        }
    )

ARCHITECTURE_DF = pd.DataFrame(
    architecture_rows
)

ARCHITECTURE_DF.to_csv(
    REPORTS_ROOT / "asff_model_architecture.csv",
    index=False,
)

display(ARCHITECTURE_DF)

print("Backbone shapes       :", backbone_shapes)
print("Detect input shapes   :", captured_detect_shapes)
print("Detect classes        :", ASFF_CORE_TEMPLATE.model[-1].nc)
print("ASFF module SHA256    :", ASFF_MODULE_SHA256)
print("YOLO26n-ASFF structural + softmax + gradient gates: PASSED")

del (
    test_core,
    neck_test,
    raw_features,
    neck_outputs,
    spatial_weights,
    PRETRAINED_DIAGNOSTIC,
    PRETRAINED_DIAGNOSTIC_CORE,
    DIAGNOSTIC_BASE_6_CORE,
)

gc.collect()
torch.cuda.empty_cache()


## J. Custom trainer

Ultralytics trainer standar dapat membangun ulang model dari YAML. Karena YAML sumber masih merepresentasikan FPN-PAN, notebook menggunakan custom trainer yang selalu mengembalikan graph ASFF yang sudah diverifikasi.

Hard gate di dalam trainer mencegah training berjalan bila `ASFFNeck` hilang, jumlah kelas berubah, atau Detect tidak lagi menerima output P3/P4/P5 dari ASFF.


In [ ]:
from ultralytics.models.yolo.detect import DetectionTrainer


class ASFFDetectionTrainer(DetectionTrainer):
    MODEL_TEMPLATE = None

    def get_model(
        self,
        cfg=None,
        weights=None,
        verbose=True,
    ):
        if self.MODEL_TEMPLATE is None:
            raise RuntimeError(
                "MODEL_TEMPLATE belum diset."
            )

        model = deepcopy(self.MODEL_TEMPLATE)

        expected_nc = int(self.data["nc"])
        actual_nc = int(model.model[-1].nc)

        if actual_nc != expected_nc:
            raise RuntimeError(
                f"Detect nc={actual_nc}, "
                f"dataset nc={expected_nc}"
            )

        if not isinstance(
            model.model[11],
            ASFFNeck,
        ):
            raise RuntimeError(
                "ASFFNeck hilang dari graph."
            )

        if model.model[-1].f != [12, 13, 14]:
            raise RuntimeError(
                "Detect source berubah: "
                f"{model.model[-1].f}"
            )

        model.names = self.data["names"]

        print("=" * 70)
        print("CUSTOM ASFF TRAINER MODEL GATE")
        print(
            "Layer 11   :",
            model.model[11].__class__.__name__,
        )
        print(
            "Detect nc  :",
            model.model[-1].nc,
        )
        print(
            "Detect from:",
            model.model[-1].f,
        )
        print("ASFF graph retained: PASSED")
        print("=" * 70)

        return model


## K. Full training research

Bagian ini menjalankan **full training**, bukan diagnostic smoke test.

Default `run_all_seeds=False` dan `active_seed=42`, sehingga satu eksekusi notebook melatih seed 42 penuh dengan konfigurasi penelitian. Untuk seed 123 dan 2026, ubah hanya `active_seed` dan jalankan notebook pada session/version terpisah.

Early stopping (`patience=25`) tetap merupakan bagian dari protokol yang sama. Karena itu jumlah epoch aktual boleh berbeda antar-seed; checkpoint evaluasi selalu `best.pt`.

Sebelum cell training ini, notebook sudah melakukan hard gate terhadap graph ASFF. Jika salah satu assertion sebelumnya gagal, **jangan lanjutkan training** sampai penyebabnya diperbaiki.


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def release_gpu_memory(reset_peak=False):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        if reset_peak:
            torch.cuda.reset_peak_memory_stats()


def safe_upload_checkpoint(
    run,
    checkpoint,
    seed,
    epoch,
    aliases,
):
    checkpoint = Path(checkpoint)

    if not WANDB_ENABLED or not checkpoint.exists():
        return False

    try:
        artifact = wandb.Artifact(
            f"yolo26n-asff-seed-{seed}-checkpoint",
            type="model",
            metadata={
                "architecture": CFG["architecture"],
                "seed": int(seed),
                "epoch": int(epoch),
                "dataset_project": CFG["project"],
                "dataset_version": CFG["dataset_version"],
                "dataset_fingerprint": DATASET_FINGERPRINT,
                "asff_module_sha256": ASFF_MODULE_SHA256,
                "checkpoint_name": checkpoint.name,
            },
        )
        artifact.add_file(str(checkpoint), name=checkpoint.name)
        artifact.add_file(str(MODULE_PATH), name="asff_modules.py")
        logged = run.log_artifact(artifact, aliases=aliases)
        logged.wait()
        return True

    except Exception as exc:
        print(
            f"[WARN] Upload {checkpoint.name} gagal: "
            f"{type(exc).__name__}: {exc}"
        )
        return False


def train_one_seed(seed):
    if not GPU_AVAILABLE:
        raise RuntimeError("GPU CUDA tidak tersedia.")

    set_seed(seed)
    release_gpu_memory(reset_peak=True)

    run_name = f"yolo26n-asff-train-seed-{seed}"
    run_dir = RUNS_ROOT / run_name

    # Hindari penghapusan hasil research tanpa sengaja.
    if run_dir.exists() and any(run_dir.iterdir()):
        raise RuntimeError(
            f"Run directory sudah berisi data: {run_dir}\n"
            "Gunakan Kaggle session/version baru atau pindahkan hasil lama "
            "sebelum mengulang seed yang sama."
        )

    run_dir.mkdir(parents=True, exist_ok=True)

    run = wandb.init(
        project=CFG["wandb_project"],
        entity=CFG["wandb_entity"],
        name=run_name,
        group="yolo26n-asff",
        job_type="research-training",
        config={
            **CFG,
            "seed": int(seed),
            "dataset_fingerprint": DATASET_FINGERPRINT,
            "asff_module_sha256": ASFF_MODULE_SHA256,
            **ENV_INFO,
        },
        mode="online" if WANDB_ENABLED else "disabled",
        reinit="finish_previous",
    )

    # Fresh seed-specific initialization for ASFF and the
    # class-specific parts of the 6-class Detect head.
    seed_template, seed_base_6 = build_seeded_asff_template(seed)

    ASFFDetectionTrainer.MODEL_TEMPLATE = seed_template

    # The wrapper is only used to enter Ultralytics train(); the custom
    # trainer replaces its graph with MODEL_TEMPLATE.
    model = YOLO(CFG["model"])

    del seed_base_6

    def on_fit_epoch_end(trainer):
        if not (WANDB_ENABLED and wandb.run):
            return

        metrics = {"epoch": int(trainer.epoch + 1)}

        for key, value in trainer.metrics.items():
            if np.isscalar(value):
                try:
                    metric_key = (
                        key if key.startswith("val/")
                        else f"val/{key}"
                    )
                    metrics[metric_key] = float(value)
                except Exception:
                    pass

        wandb.log(metrics)

    def on_model_save(trainer):
        epoch = int(trainer.epoch + 1)
        if (
            WANDB_ENABLED
            and epoch % CFG["artifact_period"] == 0
            and Path(trainer.last).exists()
        ):
            safe_upload_checkpoint(
                run,
                trainer.last,
                seed,
                epoch,
                aliases=["latest", f"epoch-{epoch}"],
            )

    model.add_callback("on_fit_epoch_end", on_fit_epoch_end)
    model.add_callback("on_model_save", on_model_save)

    started = time.perf_counter()
    status = "failed"
    caught_error = None

    try:
        model.train(
            trainer=ASFFDetectionTrainer,
            data=str(RUNTIME_YAML),
            epochs=CFG["epochs"],
            patience=CFG["patience"],
            imgsz=CFG["imgsz"],
            batch=CFG["train_batch"],
            device=DEVICE,
            workers=CFG["workers"],
            optimizer=CFG["optimizer"],
            lr0=CFG["lr0"],
            weight_decay=CFG["weight_decay"],
            warmup_epochs=CFG["warmup_epochs"],
            amp=CFG["amp"],
            seed=int(seed),
            deterministic=True,
            freeze=None,
            flipud=CFG["flipud"],
            mixup=CFG["mixup"],
            cutmix=CFG["cutmix"],
            close_mosaic=CFG["close_mosaic"],
            save=True,
            save_period=CFG["save_period"],
            plots=True,
            val=True,
            project=str(RUNS_ROOT),
            name=run_name,
            exist_ok=True,
            verbose=True,
        )
        status = "completed"

        # Post-training graph gate
        assert isinstance(model.model.model[11], ASFFNeck)
        assert model.model.model[-1].nc == DATASET_NC
        assert model.model.model[-1].f == [12, 13, 14]

    except torch.cuda.OutOfMemoryError as exc:
        caught_error = exc
        raise RuntimeError(
            "GPU OOM saat training. Jangan mengubah batch satu arsitektur "
            "secara diam-diam; revisi protokol secara konsisten bila diperlukan."
        ) from exc

    except Exception as exc:
        caught_error = exc
        raise

    finally:
        elapsed = time.perf_counter() - started
        peak_vram = (
            torch.cuda.max_memory_allocated() / 1024**2
            if torch.cuda.is_available()
            else None
        )

        trainer = getattr(model, "trainer", None)
        epochs_completed = (
            int(trainer.epoch + 1)
            if trainer is not None and hasattr(trainer, "epoch")
            else None
        )

        metadata = {
            "architecture": CFG["architecture"],
            "seed": int(seed),
            "status": status,
            "dataset_project": CFG["project"],
            "dataset_version": CFG["dataset_version"],
            "dataset_fingerprint": DATASET_FINGERPRINT,
            "asff_module_sha256": ASFF_MODULE_SHA256,
            "duration_seconds": float(elapsed),
            "training_peak_vram_mb": peak_vram,
            "epochs_completed": epochs_completed,
            "config": CFG,
            "environment": ENV_INFO,
            "error": (
                None
                if caught_error is None
                else f"{type(caught_error).__name__}: {caught_error}"
            ),
        }

        (run_dir / "run_metadata.json").write_text(
            json.dumps(metadata, indent=2),
            encoding="utf-8",
        )

        best_path = run_dir / "weights" / "best.pt"
        last_path = run_dir / "weights" / "last.pt"

        best_uploaded = safe_upload_checkpoint(
            run,
            best_path,
            seed,
            -1,
            aliases=["best", status],
        )
        last_uploaded = safe_upload_checkpoint(
            run,
            last_path,
            seed,
            -1,
            aliases=["latest", status],
        )

        print(
            "Checkpoint safe point:",
            {
                "best_exists": best_path.exists(),
                "last_exists": last_path.exists(),
                "best_wandb": best_uploaded,
                "last_wandb": last_uploaded,
            },
        )

        run.finish()
        release_gpu_memory()

    if not best_path.exists() or not last_path.exists():
        raise RuntimeError(
            f"Seed {seed} selesai tanpa best.pt/last.pt lengkap."
        )

    # Checkpoint serialization gate
    reloaded = YOLO(str(best_path))
    assert isinstance(reloaded.model.model[11], ASFFNeck)
    assert reloaded.model.model[-1].nc == DATASET_NC
    del reloaded
    release_gpu_memory()

    print(f"Seed {seed} training selesai: {run_dir}")
    return run_dir


### Full research run

Cell berikut menjalankan seed yang dipilih oleh `SEEDS_TO_RUN` menggunakan seluruh training split dan konfigurasi penelitian lengkap.

Default: `run_all_seeds=False`, `active_seed=42`, sehingga hanya seed 42 yang dilatih. Tidak ada `fraction` dan tidak ada training pendek/smoke pada bagian ini.


In [ ]:
TRAIN_DIRS = {}


for seed in SEEDS_TO_RUN:
    TRAIN_DIRS[seed] = train_one_seed(seed)

print("SAFE POINT: checkpoint training tersedia sebelum evaluasi.")


### Fairness terhadap baseline yang telah dieksekusi

Notebook mempertahankan environment, dataset fingerprint, seed, dan hyperparameter training yang sama dengan baseline/BiFPN.

Pretrained-compatible backbone dan detection head tetap berasal dari `yolo26n.pt` dengan class-name remapping yang sama seperti baseline. `ASFFNeck` sendiri merupakan modul baru sehingga diinisialisasi random secara seed-specific.

**Catatan evaluasi skala:** evaluator final menggunakan kanvas evaluasi 640×640 yang konsisten. Baseline lama yang menghitung kategori small/medium/large pada koordinat citra asli perlu dievaluasi ulang dari `best.pt`; training baseline tidak perlu diulang.


## L. Evaluasi sesuai proposal

Evaluasi dilakukan pada `best.pt` masing-masing seed.

**RQ1 / Tabel 3.5.** mAP50-95 dan mAP50 dihitung pada test set. Precision, Recall, dan F1 menggunakan satu operating point: confidence dipilih **hanya dari validation set** berdasarkan macro-F1, kemudian threshold dikunci dan diterapkan ke test set.

**RQ2 / Tabel 3.6.** AP per kelas menggunakan AP50-95. Confusion matrix Ultralytics ditampilkan sebagai diagnostik.

**RQ2 / Tabel 3.7.** AP-S/AP-M/AP-L menggunakan ambang area COCO (`32²`, `96²`) setelah ground truth dan prediksi dipetakan secara konsisten ke kanvas letterbox evaluasi `640×640`.

**RQ3 / Tabel 3.8.** Parameters, GFLOPs, actual training time di bawah protokol early stopping yang sama, model-forward latency, dan FPS. Benchmark latency menggunakan GPU, batch 1, precision yang sama, warmup yang sama, dan resolusi 640.


In [ ]:
def canonical_path(path):
    return str(Path(path).resolve())


def read_yolo_boxes(label_path, width, height):
    boxes = []
    label_path = Path(label_path)

    if not label_path.exists():
        return boxes

    for line in label_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue

        cls, xc, yc, bw, bh = map(float, line.split())
        boxes.append(
            (
                int(cls),
                (xc - bw / 2) * width,
                (yc - bh / 2) * height,
                (xc + bw / 2) * width,
                (yc + bh / 2) * height,
            )
        )

    return boxes


def image_records(split):
    df = AUDIT_DF[
        (AUDIT_DF["split"] == split)
        & (AUDIT_DF["status"].isin(NON_BLOCKING_STATUS))
    ].copy()
    return list(df.itertuples(index=False))


def predict_records(model, split, conf=None):
    """
    Prediksi seluruh split dalam chunk kecil.

    Penting:
    jangan kirim list berisi seluruh split sekaligus ke model.predict().
    Untuk eval_batch=1 setiap pemanggilan menerima satu path string;
    jika eval_batch dinaikkan, list tetap dibatasi hanya sebesar chunk.
    """
    conf = CFG["prediction_conf"] if conf is None else conf
    rows = image_records(split)

    paths = [
        str(Path(record.image).resolve())
        for record in rows
    ]

    if not paths:
        raise RuntimeError(
            f"Tidak ada image untuk split '{split}'."
        )

    batch_size = int(CFG["eval_batch"])

    if batch_size < 1:
        raise ValueError("eval_batch harus >= 1.")

    print(
        f"Predict {split}: {len(paths)} images, "
        f"chunk_size={batch_size}"
    )

    output = {}

    for start in range(0, len(paths), batch_size):
        chunk = paths[
            start : start + batch_size
        ]

        # Untuk batch=1 gunakan string path agar Ultralytics
        # tidak memperlakukan seluruh split sebagai satu batch.
        source = (
            chunk[0]
            if len(chunk) == 1
            else chunk
        )

        for result in model.predict(
            source=source,
            imgsz=CFG["imgsz"],
            conf=conf,
            iou=CFG["prediction_iou"],
            nms=False,
            device=DEVICE,
            batch=len(chunk),
            verbose=False,
            stream=True,
        ):
            boxes = result.boxes
            key = canonical_path(result.path)

            if boxes is None:
                output[key] = []
                continue

            output[key] = [
                {
                    "cls": int(cls),
                    "conf": float(score),
                    "xyxy": [
                        float(value)
                        for value in box
                    ],
                }
                for box, score, cls in zip(
                    boxes.xyxy.cpu().numpy(),
                    boxes.conf.cpu().numpy(),
                    boxes.cls.cpu().numpy(),
                )
            ]

    if len(output) != len(paths):
        raise RuntimeError(
            "Jumlah hasil prediksi tidak sama dengan jumlah image: "
            f"{len(output)} vs {len(paths)}."
        )

    return output


def iou_one_to_many(box, boxes):
    if len(boxes) == 0:
        return np.empty(0)

    boxes = np.asarray(boxes, dtype=float)
    box = np.asarray(box, dtype=float)

    intersection_size = np.maximum(
        0,
        np.minimum(box[2:], boxes[:, 2:])
        - np.maximum(box[:2], boxes[:, :2]),
    )
    intersection = intersection_size[:, 0] * intersection_size[:, 1]
    area_box = np.prod(box[2:] - box[:2])
    area_boxes = np.prod(boxes[:, 2:] - boxes[:, :2], axis=1)

    return intersection / np.maximum(
        area_box + area_boxes - intersection,
        1e-9,
    )


def match_class_boxes(
    gt_boxes,
    predictions,
    threshold,
    iou_threshold=0.5,
):
    matched = set()
    tp = 0
    fp = 0

    selected = sorted(
        [
            prediction
            for prediction in predictions
            if prediction["conf"] >= threshold
        ],
        key=lambda item: item["conf"],
        reverse=True,
    )

    for prediction in selected:
        available = [
            index
            for index in range(len(gt_boxes))
            if index not in matched
        ]
        candidate_boxes = [
            gt_boxes[index]
            for index in available
        ]
        ious = iou_one_to_many(
            prediction["xyxy"],
            candidate_boxes,
        )

        if len(ious) and ious.max() >= iou_threshold:
            matched.add(available[int(ious.argmax())])
            tp += 1
        else:
            fp += 1

    return tp, fp, len(gt_boxes) - len(matched)


def detection_counts(
    split,
    predictions,
    threshold,
    iou_threshold=0.5,
):
    counts = np.zeros((len(names), 3), dtype=int)

    for record in image_records(split):
        key = canonical_path(record.image)
        ground_truth = read_yolo_boxes(
            record.label,
            record.width,
            record.height,
        )

        for class_id in range(len(names)):
            gt_boxes = [
                item[1:]
                for item in ground_truth
                if item[0] == class_id
            ]
            class_predictions = [
                item
                for item in predictions.get(key, [])
                if item["cls"] == class_id
            ]
            counts[class_id] += match_class_boxes(
                gt_boxes,
                class_predictions,
                threshold,
                iou_threshold,
            )

    return counts


def counts_to_frame(counts):
    rows = []

    for class_id, (tp, fp, fn) in enumerate(counts):
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall
            else 0.0
        )

        rows.append(
            {
                "class_id": class_id,
                "class_name": names[class_id],
                "TP": int(tp),
                "FP": int(fp),
                "FN": int(fn),
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "ground_truth": int(tp + fn),
            }
        )

    return pd.DataFrame(rows)


def summarize_counts(counts):
    tp, fp, fn = counts.sum(axis=0)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
    }


def select_threshold(validation_predictions):
    candidates = np.linspace(0.01, 0.99, 99)
    scores = []

    for threshold in candidates:
        frame = counts_to_frame(
            detection_counts(
                "val",
                validation_predictions,
                threshold,
            )
        )
        valid = frame[frame["ground_truth"] > 0]
        scores.append(
            float(valid["f1"].mean())
            if len(valid)
            else 0.0
        )

    best_index = int(np.argmax(scores))

    return (
        float(candidates[best_index]),
        pd.DataFrame(
            {
                "confidence": candidates,
                "macro_f1": scores,
            }
        ),
    )


def save_dataframe(frame, csv_path):
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    frame.to_csv(csv_path, index=False)
    frame.to_json(
        csv_path.with_suffix(".json"),
        orient="records",
        indent=2,
        force_ascii=False,
    )


def save_figure(fig, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()


### Evaluator skala COCO pada kanvas evaluasi 640×640


In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


def to_letterbox_canvas_xyxy(
    xyxy,
    original_width,
    original_height,
    target_size,
):
    """
    Peta koordinat original-image ke kanvas square letterbox target_size.
    Transformasi yang sama diterapkan ke GT dan prediksi sehingga IoU terjaga,
    sedangkan area small/medium/large dihitung pada ukuran evaluasi yang konsisten.
    """
    scale = min(
        target_size / original_width,
        target_size / original_height,
    )

    resized_width = original_width * scale
    resized_height = original_height * scale

    pad_x = (target_size - resized_width) / 2.0
    pad_y = (target_size - resized_height) / 2.0

    x1, y1, x2, y2 = map(float, xyxy)

    mapped = np.array(
        [
            x1 * scale + pad_x,
            y1 * scale + pad_y,
            x2 * scale + pad_x,
            y2 * scale + pad_y,
        ],
        dtype=float,
    )

    mapped[[0, 2]] = np.clip(mapped[[0, 2]], 0, target_size)
    mapped[[1, 3]] = np.clip(mapped[[1, 3]], 0, target_size)

    return mapped.tolist()


def build_coco_eval_canvas(split, predictions):
    dataset = {
        "info": {},
        "licenses": [],
        "images": [],
        "annotations": [],
        "categories": [
            {"id": index + 1, "name": name}
            for index, name in enumerate(names)
        ],
    }

    results = []
    annotation_id = 1

    for image_id, record in enumerate(image_records(split), 1):
        dataset["images"].append(
            {
                "id": image_id,
                "file_name": Path(record.image).name,
                "width": CFG["imgsz"],
                "height": CFG["imgsz"],
            }
        )

        ground_truth = read_yolo_boxes(
            record.label,
            record.width,
            record.height,
        )

        for cls, x1, y1, x2, y2 in ground_truth:
            tx1, ty1, tx2, ty2 = to_letterbox_canvas_xyxy(
                [x1, y1, x2, y2],
                record.width,
                record.height,
                CFG["imgsz"],
            )

            width = max(0.0, tx2 - tx1)
            height = max(0.0, ty2 - ty1)

            dataset["annotations"].append(
                {
                    "id": annotation_id,
                    "image_id": image_id,
                    "category_id": cls + 1,
                    "bbox": [tx1, ty1, width, height],
                    "area": width * height,
                    "iscrowd": 0,
                }
            )
            annotation_id += 1

        key = canonical_path(record.image)

        for prediction in predictions.get(key, []):
            tx1, ty1, tx2, ty2 = to_letterbox_canvas_xyxy(
                prediction["xyxy"],
                record.width,
                record.height,
                CFG["imgsz"],
            )

            results.append(
                {
                    "image_id": image_id,
                    "category_id": prediction["cls"] + 1,
                    "bbox": [
                        tx1,
                        ty1,
                        max(0.0, tx2 - tx1),
                        max(0.0, ty2 - ty1),
                    ],
                    "score": prediction["conf"],
                }
            )

    return dataset, results


def ground_truth_scale_counts(dataset):
    counts = {"small": 0, "medium": 0, "large": 0}

    for annotation in dataset["annotations"]:
        area = annotation["area"]

        if area < 32**2:
            counts["small"] += 1
        elif area < 96**2:
            counts["medium"] += 1
        else:
            counts["large"] += 1

    return counts


def coco_scale_eval(dataset, results):
    counts = ground_truth_scale_counts(dataset)

    if not results:
        return {
            "AP_small": 0.0 if counts["small"] else np.nan,
            "AP_medium": 0.0 if counts["medium"] else np.nan,
            "AP_large": 0.0 if counts["large"] else np.nan,
        }

    coco_gt = COCO()
    coco_gt.dataset = dataset
    coco_gt.createIndex()

    coco_dt = coco_gt.loadRes(results)
    evaluator = COCOeval(coco_gt, coco_dt, "bbox")

    # COCO-standard maximum detections.
    evaluator.params.maxDets = [1, 10, 100]

    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    values = evaluator.stats[3:6]

    output = {}
    for key, value, count in zip(
        ["AP_small", "AP_medium", "AP_large"],
        values,
        [counts["small"], counts["medium"], counts["large"]],
    ):
        output[key] = (
            float(value)
            if count and value >= 0
            else np.nan
        )

    return output


# Boundary self-test
assert ground_truth_scale_counts(
    {
        "annotations": [
            {"area": 32**2 - 1},
            {"area": 32**2},
            {"area": 96**2 - 1},
            {"area": 96**2},
        ]
    }
) == {"small": 1, "medium": 2, "large": 1}

print("COCO scale boundary self-test: PASSED")


### Controlled efficiency benchmark


In [ ]:
def benchmark_model(model, sample_path):
    if not GPU_AVAILABLE:
        raise RuntimeError("Benchmark GPU membutuhkan CUDA.")

    release_gpu_memory(reset_peak=True)

    core = model.model.to("cuda").eval()
    use_half = bool(CFG["amp"])
    core = core.half() if use_half else core.float()

    dtype = torch.float16 if use_half else torch.float32

    dummy = torch.zeros(
        1,
        3,
        CFG["imgsz"],
        CFG["imgsz"],
        device="cuda",
        dtype=dtype,
    )

    warmup = CFG["benchmark_warmup"]
    repeats = CFG["benchmark_runs"]

    with torch.inference_mode():
        for _ in range(warmup):
            core(dummy)

        torch.cuda.synchronize()
        forward_times = []

        for _ in range(repeats):
            start = time.perf_counter()
            core(dummy)
            torch.cuda.synchronize()

            forward_times.append(
                (time.perf_counter() - start) * 1000
            )

    # Pipeline benchmark disimpan sebagai diagnostik pendukung.
    pipeline_times = []

    for _ in range(warmup):
        model.predict(
            sample_path,
            imgsz=CFG["imgsz"],
            device=0,
            half=use_half,
            nms=False,
            verbose=False,
        )

    for _ in range(repeats):
        torch.cuda.synchronize()
        start = time.perf_counter()

        model.predict(
            sample_path,
            imgsz=CFG["imgsz"],
            device=0,
            half=use_half,
            nms=False,
            verbose=False,
        )

        torch.cuda.synchronize()

        pipeline_times.append(
            (time.perf_counter() - start) * 1000
        )

    params = sum(
        parameter.numel()
        for parameter in core.parameters()
    )

    try:
        from ultralytics.utils.torch_utils import get_flops
        gflops = float(
            get_flops(
                core,
                imgsz=CFG["imgsz"],
            )
        )
    except Exception as exc:
        print(
            "[WARN] GFLOPs profiling gagal:",
            f"{type(exc).__name__}: {exc}",
        )
        gflops = np.nan

    forward_mean = float(np.mean(forward_times))
    pipeline_mean = float(np.mean(pipeline_times))

    return {
        "parameters": int(params),
        "gflops": gflops,
        "forward_latency_ms_mean": forward_mean,
        "forward_latency_ms_std": float(
            np.std(forward_times, ddof=1)
        ),
        "forward_fps": 1000.0 / forward_mean,
        "pipeline_latency_ms_mean": pipeline_mean,
        "pipeline_latency_ms_std": float(
            np.std(pipeline_times, ddof=1)
        ),
        "pipeline_fps": 1000.0 / pipeline_mean,
        "peak_vram_mb": (
            torch.cuda.max_memory_allocated() / 1024**2
        ),
        "gpu": GPU_NAME,
        "imgsz": CFG["imgsz"],
        "batch": 1,
        "precision": "FP16" if use_half else "FP32",
        "backend": "PyTorch",
        "benchmark_warmup": warmup,
        "benchmark_runs": repeats,
    }


## M. Jalankan evaluasi dan tampilkan Tabel 3.5–3.8

Bagian ini sengaja menampilkan hanya output inti proposal. File diagnostik yang lebih lengkap tetap disimpan dan diunggah ke W&B.


In [ ]:
def summarize_asff_spatial_weights(
    model,
    sample_path,
):
    """
    Ringkas bobot spasial ASFF pada satu citra diagnostik.

    Berbeda dengan BiFPN, weight ASFF bersifat input-dependent.
    Karena itu tidak ada satu vector learned weight global yang bisa
    dibaca langsung dari parameter. Hook di bawah menangkap spatial
    logits pada satu sample, mengubahnya dengan softmax, lalu merangkum
    kontribusi rata-rata source P3/P4/P5 untuk setiap target scale.
    """
    neck = model.model.model[11]

    if not isinstance(neck, ASFFNeck):
        raise TypeError(
            f"Expected ASFFNeck, got {type(neck).__name__}"
        )

    captured = {}
    handles = []

    modules = {
        "P3_target": neck.asff_p3.weight_levels,
        "P4_target": neck.asff_p4.weight_levels,
        "P5_target": neck.asff_p5.weight_levels,
    }

    def make_hook(label):
        def hook(module, inputs, output):
            weights = torch.softmax(
                output.detach().float(),
                dim=1,
            )
            captured[label] = weights.cpu()
        return hook

    for label, module in modules.items():
        handles.append(
            module.register_forward_hook(
                make_hook(label)
            )
        )

    try:
        _ = model.predict(
            str(sample_path),
            imgsz=CFG["imgsz"],
            conf=CFG["prediction_conf"],
            iou=CFG["prediction_iou"],
            nms=False,
            device=DEVICE,
            batch=1,
            verbose=False,
        )
    finally:
        for handle in handles:
            handle.remove()

    rows = []
    json_summary = {}

    source_labels = ["P3", "P4", "P5"]

    for target, weights in captured.items():
        # [B, 3, H, W] -> 3 source means.
        means = weights.mean(
            dim=(0, 2, 3)
        ).numpy()

        stds = weights.std(
            dim=(0, 2, 3),
            unbiased=False,
        ).numpy()

        target_summary = {}

        for source, mean_value, std_value in zip(
            source_labels,
            means,
            stds,
        ):
            rows.append(
                {
                    "target_scale": target,
                    "source_scale": source,
                    "mean_weight": float(mean_value),
                    "std_weight": float(std_value),
                }
            )

            target_summary[source] = {
                "mean": float(mean_value),
                "std": float(std_value),
            }

        sum_error = (
            weights.sum(dim=1) - 1.0
        ).abs().max().item()

        target_summary["max_softmax_sum_error"] = float(
            sum_error
        )

        json_summary[target] = target_summary

    if len(captured) != 3:
        raise RuntimeError(
            "ASFF diagnostic hook tidak menangkap ketiga target scale."
        )

    return pd.DataFrame(rows), json_summary


def upload_evaluation_to_wandb(
    seed,
    eval_dir,
    overall,
):
    if not WANDB_ENABLED:
        return {
            "seed": int(seed),
            "status": "wandb_disabled",
        }

    run = None

    try:
        run = wandb.init(
            project=CFG["wandb_project"],
            entity=CFG["wandb_entity"],
            name=f"yolo26n-asff-evaluation-seed-{seed}",
            group="yolo26n-asff",
            job_type="research-evaluation",
            config={
                **CFG,
                "seed": int(seed),
                "dataset_fingerprint": DATASET_FINGERPRINT,
                "asff_module_sha256": ASFF_MODULE_SHA256,
            },
            reinit="finish_previous",
        )

        scalar_metrics = {
            key: value
            for key, value in overall.items()
            if (
                isinstance(value, (int, float))
                and not pd.isna(value)
            )
        }

        run.log(
            {
                f"test/{key}": value
                for key, value in scalar_metrics.items()
            }
        )

        for csv_path in sorted(Path(eval_dir).glob("*.csv")):
            run.log(
                {
                    f"tables/{csv_path.stem}":
                        wandb.Table(
                            dataframe=pd.read_csv(csv_path)
                        )
                }
            )

        figure_dir = Path(eval_dir) / "figures"

        if figure_dir.exists():
            for image_path in sorted(figure_dir.glob("*.png")):
                run.log(
                    {
                        f"figures/{image_path.stem}":
                            wandb.Image(str(image_path))
                    }
                )

        artifact = wandb.Artifact(
            f"yolo26n-asff-evaluation-seed-{seed}",
            type="evaluation",
            metadata={
                "architecture": CFG["architecture"],
                "seed": int(seed),
                "dataset_fingerprint": DATASET_FINGERPRINT,
                "asff_module_sha256": ASFF_MODULE_SHA256,
            },
        )

        artifact.add_dir(str(eval_dir))
        artifact.add_file(
            str(MODULE_PATH),
            name="asff_modules.py",
        )

        logged = run.log_artifact(
            artifact,
            aliases=["latest", f"seed-{seed}"],
        )
        logged.wait()

        return {
            "seed": int(seed),
            "status": "uploaded",
            "run_url": run.url,
        }

    except Exception as exc:
        print(
            f"[WARN] Upload evaluasi seed {seed} gagal: "
            f"{type(exc).__name__}: {exc}"
        )
        return {
            "seed": int(seed),
            "status": "failed",
            "error": f"{type(exc).__name__}: {exc}",
        }

    finally:
        if run is not None:
            run.finish()


def evaluate_seed(seed, run_dir):
    run_dir = Path(run_dir)
    best_path = run_dir / "weights" / "best.pt"

    if not best_path.exists():
        raise FileNotFoundError(best_path)

    release_gpu_memory()
    model = YOLO(str(best_path))

    assert isinstance(model.model.model[11], ASFFNeck)
    assert model.model.model[-1].nc == DATASET_NC

    eval_dir = run_dir / "evaluation"
    figure_dir = eval_dir / "figures"
    figure_dir.mkdir(parents=True, exist_ok=True)

    # ========================================================
    # 1. Validation selects confidence threshold
    # ========================================================

    val_predictions = predict_records(
        model,
        "val",
        conf=CFG["prediction_conf"],
    )

    best_confidence, threshold_curve = (
        select_threshold(val_predictions)
    )

    save_dataframe(
        threshold_curve,
        eval_dir / "validation_threshold_curve.csv",
    )

    del val_predictions
    release_gpu_memory()

    # ========================================================
    # 2. Test predictions at low confidence
    # ========================================================

    test_predictions = predict_records(
        model,
        "test",
        conf=CFG["prediction_conf"],
    )

    test_counts = detection_counts(
        "test",
        test_predictions,
        best_confidence,
    )

    test_per_class_operating = counts_to_frame(
        test_counts
    )

    count_summary = summarize_counts(
        test_counts
    )

    save_dataframe(
        test_per_class_operating,
        eval_dir / "test_operating_point_per_class.csv",
    )

    # ========================================================
    # 3. Ultralytics AP / mAP on test
    # ========================================================

    release_gpu_memory()

    metrics = model.val(
        data=str(RUNTIME_YAML),
        split="test",
        imgsz=CFG["imgsz"],
        batch=CFG["eval_batch"],
        conf=CFG["prediction_conf"],
        iou=CFG["prediction_iou"],
        nms=False,
        device=DEVICE,
        plots=True,
        project=str(eval_dir),
        name="ultralytics",
        exist_ok=True,
        verbose=False,
    )

    ultralytics_dir = Path(metrics.save_dir)

    ap50_all = np.full(
        len(names),
        np.nan,
        dtype=float,
    )

    ap_class_index = np.asarray(
        metrics.box.ap_class_index,
        dtype=int,
    )

    ap50_present = np.asarray(
        metrics.box.ap50,
        dtype=float,
    )

    if len(ap_class_index) == len(ap50_present):
        ap50_all[ap_class_index] = ap50_present

    ap50_95_all = np.asarray(
        metrics.box.maps,
        dtype=float,
    )

    if len(ap50_95_all) != len(names):
        fallback = np.full(
            len(names),
            np.nan,
            dtype=float,
        )

        ap_present = np.asarray(
            metrics.box.ap,
            dtype=float,
        )

        if len(ap_class_index) == len(ap_present):
            fallback[ap_class_index] = ap_present

        ap50_95_all = fallback

    per_class_ap = pd.DataFrame(
        {
            "class_id": range(len(names)),
            "class_name": names,
            "AP50": ap50_all,
            "AP50_95": ap50_95_all,
        }
    ).merge(
        test_per_class_operating[
            ["class_id", "ground_truth"]
        ],
        on="class_id",
        how="left",
    )

    save_dataframe(
        per_class_ap,
        eval_dir / "test_per_class_ap.csv",
    )

    # ========================================================
    # RQ1 / Proposal Table 3.5
    # ========================================================

    overall = {
        "architecture": CFG["architecture"],
        "model": CFG["model_name"],
        "seed": int(seed),
        "confidence_from_validation": float(best_confidence),
        "mAP50_95": float(metrics.box.map),
        "mAP50": float(metrics.box.map50),
        **count_summary,
        "dataset_fingerprint": DATASET_FINGERPRINT,
    }

    table_35 = pd.DataFrame(
        [
            {
                "Model": CFG["model_name"],
                "Seed": int(seed),
                "mAP50-95": overall["mAP50_95"],
                "mAP50": overall["mAP50"],
                "Precision": overall["precision"],
                "Recall": overall["recall"],
                "F1-score": overall["f1"],
            }
        ]
    )

    save_dataframe(
        table_35,
        eval_dir / "tabel_3_5_kinerja_keseluruhan.csv",
    )

    # ========================================================
    # RQ2 / Proposal Table 3.6
    # ========================================================

    table_36 = (
        per_class_ap
        .set_index("class_name")["AP50_95"]
        .reindex(names)
        .to_frame()
        .T
    )

    table_36.insert(
        0,
        "Model",
        CFG["model_name"],
    )

    table_36.insert(
        1,
        "Seed",
        int(seed),
    )

    save_dataframe(
        table_36,
        eval_dir / "tabel_3_6_ap_per_kelas.csv",
    )

    # Threshold-selection figure
    fig, ax = plt.subplots(figsize=(8, 5))

    ax.plot(
        threshold_curve["confidence"],
        threshold_curve["macro_f1"],
    )

    ax.axvline(
        best_confidence,
        linestyle="--",
        label=f"Terpilih = {best_confidence:.2f}",
    )

    ax.set(
        title="Pemilihan Confidence dari Validation Set",
        xlabel="Confidence",
        ylabel="Macro F1",
    )

    ax.grid(alpha=0.25)
    ax.legend()

    save_figure(
        fig,
        figure_dir / "validation_macro_f1_threshold.png",
    )

    # Per-class AP figure
    fig, ax = plt.subplots(figsize=(11, 5))

    ax.bar(
        per_class_ap["class_name"],
        per_class_ap["AP50_95"],
    )

    ax.set(
        title="AP50-95 per Kelas — Test Set",
        xlabel="Kelas",
        ylabel="AP50-95",
        ylim=(0, 1),
    )

    ax.tick_params(
        axis="x",
        rotation=25,
    )

    ax.grid(
        axis="y",
        alpha=0.25,
    )

    save_figure(
        fig,
        figure_dir / "ap50_95_per_kelas.png",
    )

    # Copy and display confusion matrices as diagnostics
    confusion_paths = []

    for filename in [
        "confusion_matrix.png",
        "confusion_matrix_normalized.png",
    ]:
        source = ultralytics_dir / filename

        if source.exists():
            target = figure_dir / filename
            shutil.copy2(source, target)
            confusion_paths.append(target)

    # ========================================================
    # RQ2 / Proposal Table 3.7 — scale
    # ========================================================

    coco_gt, coco_predictions = build_coco_eval_canvas(
        "test",
        test_predictions,
    )

    scale_metrics = coco_scale_eval(
        coco_gt,
        coco_predictions,
    )

    overall.update(scale_metrics)

    (eval_dir / "coco_scale_metrics.json").write_text(
        json.dumps(
            scale_metrics,
            indent=2,
            allow_nan=True,
        ),
        encoding="utf-8",
    )

    table_37 = pd.DataFrame(
        [
            {
                "Model": CFG["model_name"],
                "Seed": int(seed),
                "AP-S": scale_metrics["AP_small"],
                "AP-M": scale_metrics["AP_medium"],
                "AP-L": scale_metrics["AP_large"],
            }
        ]
    )

    save_dataframe(
        table_37,
        eval_dir / "tabel_3_7_ap_berdasarkan_skala.csv",
    )

    fig, ax = plt.subplots(figsize=(7, 5))

    scale_values = [
        scale_metrics["AP_small"],
        scale_metrics["AP_medium"],
        scale_metrics["AP_large"],
    ]

    bars = ax.bar(
        ["Small", "Medium", "Large"],
        scale_values,
    )

    ax.set(
        title="AP50-95 Berdasarkan Skala Objek",
        ylabel="AP50-95",
        ylim=(0, 1),
    )

    ax.grid(
        axis="y",
        alpha=0.25,
    )

    for bar, value in zip(bars, scale_values):
        if not pd.isna(value):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                value,
                f"{value:.3f}",
                ha="center",
                va="bottom",
            )

    save_figure(
        fig,
        figure_dir / "ap_berdasarkan_skala.png",
    )

    # ========================================================
    # RQ3 / Proposal Table 3.8 — efficiency
    # ========================================================

    sample_path = image_records("test")[0].image

    efficiency = benchmark_model(
        model,
        sample_path,
    )

    metadata = json.loads(
        (run_dir / "run_metadata.json")
        .read_text(encoding="utf-8")
    )

    efficiency["training_duration_seconds"] = (
        metadata["duration_seconds"]
    )

    efficiency["training_peak_vram_mb"] = (
        metadata.get("training_peak_vram_mb")
    )

    efficiency["checkpoint_mb"] = (
        best_path.stat().st_size / 1024**2
    )

    overall.update(efficiency)

    (eval_dir / "efficiency_full.json").write_text(
        json.dumps(
            efficiency,
            indent=2,
            allow_nan=True,
        ),
        encoding="utf-8",
    )

    table_38 = pd.DataFrame(
        [
            {
                "Model": CFG["model_name"],
                "Seed": int(seed),
                "Parameters (juta)":
                    efficiency["parameters"] / 1e6,
                "GFLOPs":
                    efficiency["gflops"],
                "Waktu training (jam)":
                    efficiency["training_duration_seconds"] / 3600,
                "Latency (ms/citra)":
                    efficiency["forward_latency_ms_mean"],
                "FPS":
                    efficiency["forward_fps"],
            }
        ]
    )

    save_dataframe(
        table_38,
        eval_dir / "tabel_3_8_efisiensi_komputasi.csv",
    )

    # Supporting diagnostics: not part of proposal table.
    efficiency_diagnostic = pd.DataFrame(
        [
            {
                "Seed": int(seed),
                "Model size (MB)": efficiency["checkpoint_mb"],
                "Peak VRAM training (MB)":
                    efficiency["training_peak_vram_mb"],
                "Peak VRAM inferensi (MB)":
                    efficiency["peak_vram_mb"],
                "Pipeline latency (ms/citra)":
                    efficiency["pipeline_latency_ms_mean"],
                "Pipeline FPS":
                    efficiency["pipeline_fps"],
                "GPU":
                    efficiency["gpu"],
                "Precision":
                    efficiency["precision"],
                "Resolusi":
                    efficiency["imgsz"],
            }
        ]
    )

    save_dataframe(
        efficiency_diagnostic,
        eval_dir / "efficiency_supporting_diagnostics.csv",
    )

    # ASFF spatial-weight diagnostic on one held-out test image.
    diagnostic_sample = image_records("test")[0].image

    asff_weight_df, asff_weight_json = (
        summarize_asff_spatial_weights(
            model,
            diagnostic_sample,
        )
    )

    save_dataframe(
        asff_weight_df,
        eval_dir / "asff_spatial_weight_summary.csv",
    )

    (eval_dir / "asff_spatial_weight_summary.json").write_text(
        json.dumps(
            asff_weight_json,
            indent=2,
        ),
        encoding="utf-8",
    )

    # Overall JSON after all sections
    (eval_dir / "overall_metrics.json").write_text(
        json.dumps(
            overall,
            indent=2,
            allow_nan=True,
        ),
        encoding="utf-8",
    )

    # ========================================================
    # DISPLAY — proposal outputs only
    # ========================================================

    display(
        Markdown(
            f"### Seed {seed} — Tabel 3.5: "
            "Kinerja Deteksi Secara Keseluruhan"
        )
    )
    display(table_35.style.format(precision=4))

    display(
        Markdown(
            f"### Seed {seed} — Tabel 3.6: "
            "AP50-95 Berdasarkan Kelas"
        )
    )
    display(table_36.style.format(precision=4))

    if confusion_paths:
        display(
            Markdown(
                "### Confusion Matrix — diagnostik pendukung RQ2"
            )
        )
        for path in confusion_paths:
            display(IPyImage(filename=str(path)))

    display(
        Markdown(
            f"### Seed {seed} — Tabel 3.7: "
            "AP Berdasarkan Skala Objek"
        )
    )
    display(table_37.style.format(precision=4))

    display(
        Markdown(
            f"### Seed {seed} — Tabel 3.8: "
            "Efisiensi Komputasi"
        )
    )
    display(table_38.style.format(precision=4))

    display(
        Markdown(
            "### Diagnostik ASFF — mean spatial weights pada 1 citra test"
        )
    )
    display(
        asff_weight_df.style.format(precision=4)
    )

    # ========================================================
    # W&B evaluation upload
    # ========================================================

    upload_status = upload_evaluation_to_wandb(
        seed,
        eval_dir,
        overall,
    )

    (eval_dir / "wandb_upload_status.json").write_text(
        json.dumps(
            upload_status,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        f"Seed {seed}: evaluation complete | "
        f"W&B status: {upload_status['status']}"
    )

    del test_predictions, model
    release_gpu_memory()

    return {
        "seed": int(seed),
        "run_dir": str(run_dir),
        "eval_dir": str(eval_dir),
        **overall,
    }


EVALUATION_SUMMARIES = []

for seed, run_dir in TRAIN_DIRS.items():
    EVALUATION_SUMMARIES.append(
        evaluate_seed(
            seed,
            run_dir,
        )
    )


## N. Ringkasan seed yang selesai

Jika notebook menjalankan lebih dari satu seed pada session yang sama, tabel agregat mean ± sample standard deviation juga dibuat. Untuk default satu-seed-per-run, hasil tiap seed tetap tersimpan di W&B dan nantinya digabung pada notebook final comparison.


In [ ]:
SEED_RESULTS = pd.DataFrame(
    EVALUATION_SUMMARIES
)

SEED_RESULTS.to_csv(
    REPORTS_ROOT / "asff_seed_results.csv",
    index=False,
)

display(
    SEED_RESULTS[
        [
            "seed",
            "mAP50_95",
            "mAP50",
            "precision",
            "recall",
            "f1",
            "AP_small",
            "AP_medium",
            "AP_large",
            "parameters",
            "gflops",
            "forward_latency_ms_mean",
            "forward_fps",
        ]
    ].style.format(precision=4)
)

if len(SEED_RESULTS) >= 2:
    aggregate_columns = [
        "mAP50_95",
        "mAP50",
        "precision",
        "recall",
        "f1",
        "AP_small",
        "AP_medium",
        "AP_large",
        "forward_latency_ms_mean",
        "forward_fps",
    ]

    AGGREGATE = pd.DataFrame(
        {
            "metric": aggregate_columns,
            "mean": [
                SEED_RESULTS[column].mean()
                for column in aggregate_columns
            ],
            "sample_std": [
                SEED_RESULTS[column].std(ddof=1)
                for column in aggregate_columns
            ],
            "n_seed": len(SEED_RESULTS),
        }
    )

    AGGREGATE.to_csv(
        REPORTS_ROOT / "asff_seed_mean_std.csv",
        index=False,
    )

    display(
        Markdown(
            "### Ringkasan ASFF antar-seed — mean ± sample SD"
        )
    )
    display(AGGREGATE.style.format(precision=4))
else:
    print(
        "Aggregate mean±SD belum ditampilkan karena session ini "
        "hanya memuat satu seed."
    )


## O. Final gate

Pipeline dianggap lengkap bila:

- dataset audit dan provenance V2 lolos;
- custom ASFF module memiliki hash yang tercatat;
- arsitektur dan gradient gate lolos;
- `best.pt` dan `last.pt` tersedia;
- `best.pt` dapat direload sebagai YOLO26n-ASFF;
- Tabel 3.5, 3.6, 3.7, dan 3.8 tersedia untuk setiap seed yang dijalankan;
- evaluation artifact dicoba diunggah ke W&B.

**Friedman Test belum dilakukan di sini.** Uji statistik dilakukan pada notebook final comparison setelah 4 arsitektur × 3 seed lengkap.


In [ ]:
required_global = [
    REPORTS_ROOT / "dataset_audit.csv",
    REPORTS_ROOT / "asff_model_architecture.csv",
    MODULE_PATH,
]

for path in required_global:
    if not Path(path).exists():
        raise RuntimeError(
            f"File global wajib tidak ditemukan: {path}"
        )

for seed, run_dir in TRAIN_DIRS.items():
    run_dir = Path(run_dir)

    required_seed = [
        run_dir / "weights" / "best.pt",
        run_dir / "weights" / "last.pt",
        run_dir / "run_metadata.json",
        run_dir / "evaluation" / "overall_metrics.json",
        run_dir / "evaluation" / "tabel_3_5_kinerja_keseluruhan.csv",
        run_dir / "evaluation" / "tabel_3_6_ap_per_kelas.csv",
        run_dir / "evaluation" / "tabel_3_7_ap_berdasarkan_skala.csv",
        run_dir / "evaluation" / "tabel_3_8_efisiensi_komputasi.csv",
        run_dir / "evaluation" / "asff_spatial_weight_summary.json",
    ]

    missing = [
        str(path)
        for path in required_seed
        if not path.exists()
    ]

    if missing:
        raise RuntimeError(
            "Pipeline belum lengkap untuk seed "
            f"{seed}:\n" + "\n".join(missing)
        )

print("=" * 72)
print("END-TO-END YOLO26n-ASFF PIPELINE PASSED")
print("Dataset fingerprint:", DATASET_FINGERPRINT)
print("Module fingerprint :", ASFF_MODULE_SHA256)
print("Seeds completed    :", list(TRAIN_DIRS))
print("=" * 72)


## Catatan penggunaan notebook clean

Setelah seed 42 selesai end-to-end:

1. simpan `best.pt`, metadata training, dan evaluation artifact;
2. periksa kurva training, test metrics, AP-S/M/L, per-class AP, efisiensi, dan spatial-weight diagnostic;
3. untuk menjalankan seed `123` atau `2026`, ubah **hanya** `CFG['active_seed']` pada notebook yang sama;
4. jangan mengubah arsitektur ASFF atau hyperparameter di tengah rangkaian tiga seed;
5. Friedman Test dilakukan belakangan setelah seluruh 4 arsitektur × 3 seed lengkap.

Evaluator pada notebook ini menggunakan sumber prediksi yang diproses secara terkontrol untuk mencegah pola OOM yang sebelumnya muncul ketika seluruh daftar image path diberikan sekaligus.
